In [16]:
import pandas as pd
import numpy as np
import plotly.express as px

# importerer data
weather_df = pd.read_csv("../raw_data/weather.csv")

In [17]:
# første 5 radene
head = weather_df.head()
head

,timestamp,temperature,precipitation,wind_speed
0,2023-12-31 23:00:00+00:00,3.0,0.0,17.9
1,2024-01-01 00:00:00+00:00,2.8,0.0,13.2
2,2024-01-01 01:00:00+00:00,2.6,0.0,10.5
3,2024-01-01 02:00:00+00:00,2.7,0.0,18.9
4,2024-01-01 03:00:00+00:00,2.1,0.0,18.4


In [18]:
# siste 5 radene
tail = weather_df.tail()
tail

,timestamp,temperature,precipitation,wind_speed
11702,2025-05-02 12:00:00+00:00,8.8,1.7,17.0
11703,2025-05-02 13:00:00+00:00,10.5,0.1,20.2
11704,2025-05-02 14:00:00+00:00,9.9,1.4,17.6
11705,2025-05-02 15:00:00+00:00,10.1,0.1,16.6
11706,2025-05-02 16:00:00+00:00,9.1,1.1,16.9


In [19]:
# korrigerer til riktig tidssone før videre analyse
# PS - vil endre dtype på timestamp til Europe/Oslo
weather_df["timestamp"] = pd.to_datetime(weather_df["timestamp"], utc=True).dt.tz_convert("Europe/Oslo")

# evt - viss datetime ikke er viktig - kan være aktuelt mtp at alle tidene er riktig formatert allerede
#weather_df["timestamp"] = weather_df["timestamp"].dt.tz_convert("Europe/Oslo")

In [20]:
# data info
info = weather_df.info()
info

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11707 entries, 0 to 11706
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype                      
---  ------         --------------  -----                      
 0   timestamp      11707 non-null  datetime64[ns, Europe/Oslo]
 1   temperature    11575 non-null  float64                    
 2   precipitation  11599 non-null  float64                    
 3   wind_speed     11597 non-null  float64                    
dtypes: datetime64[ns, Europe/Oslo](1), float64(3)
memory usage: 366.0 KB


In [21]:
# data beskrivelse
description = weather_df.describe(include='all')
description

,timestamp,temperature,precipitation,wind_speed
count,11707,11575.000000,11599.000000,11597.000000
mean,2024-08-31 21:30:01.998803968+02:00,7.054609,0.318036,11.352358
min,2024-01-01 00:00:00+01:00,-16.000000,0.000000,0.000000
25%,2024-05-01 22:30:00+02:00,2.900000,0.000000,5.600000
50%,2024-08-31 21:00:00+02:00,6.900000,0.000000,9.900000
75%,2024-12-31 19:30:00+01:00,11.400000,0.200000,15.700000
max,2025-05-02 18:00:00+02:00,27.900000,9.600000,40.600000
std,NaN,6.521379,0.743150,7.161594


In [22]:
# hvor lenge målingene har holdt på
min = weather_df["timestamp"].min()
max = weather_df["timestamp"].max()

print("Målinger startet", min, "og sluttet", max)

Målinger startet 2024-01-01 00:00:00+01:00 og sluttet 2025-05-02 18:00:00+02:00


In [23]:
# sjekk for manglende verdier
isna = weather_df.isna().sum()
isna

timestamp          0
temperature      132
precipitation    108
wind_speed       110
dtype: int64

In [24]:
# sjekk for unike rader
nunique = weather_df.nunique()
nunique

timestamp        11705
temperature        409
precipitation       69
wind_speed         374
dtype: int64

In [35]:
# sjekker etter outliers / unormale verdier i data
outliers = weather_df.query("temperature < -40 or temperature > 40 or wind_speed < 0 or wind_speed > 45")
if outliers.empty:
    print("Ingen outliers / unormale verdier i data")

Ingen outliers / unormale verdier i data


In [25]:
# finner laveste og høyeste målte temperaturer
lowest_temp = weather_df["temperature"].min()
highest_temp = weather_df["temperature"].max()
lowest_temp_time = weather_df[weather_df["temperature"] == lowest_temp]["timestamp"].dt.strftime("%d.%m.%y %H:%M").iloc[0]
highest_temp_time = weather_df[weather_df["temperature"] == highest_temp]["timestamp"].dt.strftime("%d.%m.%y %H:%M").iloc[0]

print("Laveste temperatur var", lowest_temp, "C, og inntraff", lowest_temp_time,
       "\nHøyeste temperatur var", highest_temp, "C, og inntraff", highest_temp_time)

Laveste temperatur var -16.0 C, og inntraff 18.01.24 02:00 
Høyeste temperatur var 27.9 C, og inntraff 05.09.24 16:00


In [26]:
# finner laveste og høyeste målte vindfart
lowest_wind = weather_df["wind_speed"].min()
highest_wind = weather_df["wind_speed"].max()
lowest_wind_time = weather_df[weather_df["wind_speed"] == lowest_wind]["timestamp"].dt.strftime("%d.%m.%y %H:%M").iloc[0]
highest_wind_time = weather_df[weather_df["wind_speed"] == highest_wind]["timestamp"].dt.strftime("%d.%m.%y %H:%M").iloc[0]

print("Laveste vindfart var", lowest_wind, "m/s, og inntraff først i målingene", lowest_wind_time,
       "\nHøyeste vindfart var", highest_wind, "m/s, og inntraff", highest_wind_time)

Laveste vindfart var 0.0 m/s, og inntraff først i målingene 27.04.24 10:00 
Høyeste vindfart var 40.6 m/s, og inntraff 21.10.24 08:00


In [27]:
# visualiserer temperatur over tid via lineplot
fig = px.line(weather_df, "timestamp", "temperature", title="temperatur over tid")
fig.show()

In [28]:
# visualiserer vindfart over tid via lineplot
fig = px.line(weather_df, "timestamp", "wind_speed", title="vindfart over tid")
fig.show()

In [ ]:
# visualiserer om været varierer særlig mellom dager i uka via boxplot
weather_df["day_of_week"] = weather_df["timestamp"].dt.day_name()

fig = px.box(weather_df, x="day_of_week", y="temperature", title="temperaturforskjeller mellom dager i uka")
fig.update_layout(xaxis_title="dag i uka", yaxis_title="temperatur")
fig.show()

In [39]:
# visualiserer om været varierer særlig mellom dager i uka via boxplot
weather_df["month"] = weather_df["timestamp"].dt.month_name()

fig = px.box(weather_df, x="month", y="temperature", title="temperaturforskjeller mellom måneder i året")
fig.update_layout(xaxis_title="måned", yaxis_title="temperatur")
fig.show()

In [40]:
# visualiserer temperatur og vindfart via scatter
fig = px.scatter(weather_df, "temperature", "wind_speed")
fig.show()